In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Wizualizacja czasu wykonania obwodu

<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy używanie tych lub nowszych wersji.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Chociaż [rysownik osi czasu](/guides/visualize-circuit-timing) wbudowany w Qiskit jest przydatny dla statycznych obwodów, może nie dokładnie odzwierciedlać czasu wykonania [obwodów dynamicznych](/guides/classical-feedforward-and-control-flow) ze względu na niejawne operacje, takie jak rozgłaszanie i określanie gałęzi. W ramach obsługi obwodów dynamicznych Qiskit Runtime zwraca dokładne informacje o czasie wykonania obwodu wewnątrz wyników zadania, gdy jest to wymagane.

> **Note:** - Jest to funkcja eksperymentalna. Jest w fazie wersji zapoznawczej i dlatego może ulec zmianie.
> - Ta funkcja dotyczy tylko zadań Qiskit Runtime Sampler.
> - Chociaż całkowity czas obwodu jest zwracany w metadanych "compilation", NIE jest to czas używany do rozliczeń (czas QPU).
### Włączanie pobierania danych czasowych
Aby włączyć pobieranie danych czasowych, ustaw eksperymentalną flagę `scheduler_timing` na `True` podczas uruchamiania zadania prymitywu.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Dostęp do danych czasowych obwodu
Gdy jest to wymagane, dane czasowe obwodu dla każdego PUB są zwracane w metadanych wynikowych zadania, pod `["compilation"]["scheduler_timing"]["timing"]`. To pole zawiera surowe informacje o czasie. Aby wyświetlić informacje o czasie, użyj wbudowanego narzędzia wizualizacji, jak opisano w sekcji [Wizualizacja czasów](#visualize-timings).

Użyj poniższego kodu, aby uzyskać dostęp do danych czasowych obwodu dla pierwszego PUB:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Zrozumienie surowych danych czasowych
Chociaż wizualizacja danych czasowych obwodu za pomocą metody `draw_circuit_schedule_timing` jest najczęstszym przypadkiem użycia, może być przydatne zrozumienie struktury zwracanych surowych danych czasowych. Może to pomóc, na przykład, w programowym wyodrębnianiu informacji.

Dane czasowe zwracane w `["compilation"]["scheduler_timing"]["timing"]` to lista ciągów znaków. Każdy ciąg reprezentuje pojedynczą instrukcję na jakimś kanale i jest rozdzielony przecinkami na następujące typy danych:

- `Branch` - Określa, czy instrukcja jest w przepływie sterowania (then / else) czy w gałęzi głównej.
- `Instruction` - Bramka i kubit, na którym ma działać.
- `Channel` - Kanał, do którego przypisana jest instrukcja. Może być jednym z następujących:
      - `Qubit x` - Kanał sterowania dla kubitu _x_.
      - `AWGRx_y` (generator dowolnych przebiegów do odczytu) - Używany przez kanały odczytu do komunikacji podczas pomiaru kubitów. Argumenty _x_ i _y_ odpowiadają odpowiednio identyfikatorowi instrumentu odczytu i numerowi kubitu.
- `T0` - Czas rozpoczęcia instrukcji w pełnym harmonogramie
- `Duration` - Czas trwania instrukcji, w jednostkach _dt_ sekund, gdzie 1 dt = 1 cykl harmonogramowania. Wartość `dt` backendu można znaleźć za pomocą [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - Typ używanej operacji impulsowej.

Przykład:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Wizualizacja czasów
W wersji `qiskit-ibm-runtime` v0.43.0 lub nowszej możesz wizualizować czasy obwodów. Aby zwizualizować czasy, musisz najpierw przekonwertować metadane wynikowe na `fig` za pomocą [metody `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). Ta metoda zwraca figurę `plotly`, którą możesz wyświetlić bezpośrednio, zapisać do pliku lub zrobić jedno i drugie. Aby uzyskać więcej informacji o poleceniach `plotly`, zobacz [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) i [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Najechanie kursorem na wyjście pokazuje informacje takie jak czas rozpoczęcia, zakończenia i czas trwania.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Przykład wygenerowanej figury')
#### Zrozumienie wygenerowanej figury
Obraz wyjściowy danych czasowych obwodu generowany przez `draw_circuit_schedule_timing` przekazuje następujące informacje:

- Oś X to czas w jednostkach _dt_ sekund, gdzie 1 dt = 1 cykl harmonogramowania. Wartość `dt` backendu można znaleźć za pomocą [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Oś Y to kanał (kanały można traktować jako instrumenty emitujące impulsy).
    - `Receive channel` - Jedyny kanał, który sam w sobie nie jest instrumentem. Jest to instrukcja odtwarzana na wszystkich kanałach, które są częścią procedury komunikacji z centrum w danym czasie.
    - `Qubit x` - Kanał sterowania dla kubitu x.
    - `AWGRx_y` (generator dowolnych przebiegów do odczytu) - Używany przez kanały odczytu do komunikacji podczas pomiaru kubitów. Argumenty _x_ i _y_ odpowiadają odpowiednio identyfikatorowi instrumentu odczytu i numerowi kubitu.
    - `Hub` - Kontroluje rozgłaszanie.

Ponadto każda instrukcja ma format *X_Y*, gdzie *X* to nazwa instrukcji, a *Y* to typ impulsu. Typ `play` stosuje impulsy sterujące, a `capture` rejestruje stan kubitu. Możesz również najechać kursorem na każdą instrukcję, aby uzyskać więcej szczegółów. Na przykład poprzednia figura pokazuje impuls sterujący dla bramki X zastosowany do kubitu 10 w czasie 1161 dt.
### Przykład end-to-end
Ten przykład pokazuje, jak włączyć opcję, uzyskać informacje o czasie obwodu z metadanych i wyświetlić je jako obraz.

Najpierw skonfiguruj środowisko, zdefiniuj obwody i przekonwertuj je na obwody ISA, a następnie zdefiniuj i uruchom zadania.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Następnie pobierz harmonogram czasu obwodu: